# Controlled strategy experiments

This notebook demonstrates a chronological train/validation/test experiment. Training results are diagnostic, validation selects the strategy, and the final test interval is evaluated once.

In [ ]:
import numpy as np
import pandas as pd

from cobasket.strategy_experiments import (
    ExperimentSplit,
    StrategyExperimentConfig,
    run_strategy_experiment,
)
from cobasket.strategy_rules import MetricCondition, StrategyRule, StrategyRules

## Synthetic prices and metrics

The example is deliberately synthetic so the notebook works offline. In a real run, `probability`, `stable`, `momentum_score`, `trend_score`, and `high_volatility` would come from Cobasket's walk-forward and price-metric modules.

In [ ]:
rng = np.random.default_rng(42)
dates = pd.date_range('2018-01-01', periods=750, freq='B')
returns = rng.normal(0.0003, 0.012, size=(len(dates), 3))
prices = pd.DataFrame(100 * np.exp(np.cumsum(returns, axis=0)), index=dates, columns=['AAA', 'BBB', 'CCC'])

probability = pd.DataFrame(rng.uniform(0.25, 0.75, size=prices.shape), index=dates, columns=prices.columns)
momentum = prices.pct_change(40).clip(-0.2, 0.2) / 0.2
stable = pd.DataFrame(True, index=dates, columns=prices.columns)
metrics = {'probability': probability, 'momentum_score': momentum, 'stable': stable.astype(float)}

## Declare candidate strategies

The candidates differ only in whether positive momentum is required. Rule order remains explicit: sell conditions are evaluated before buy conditions.

In [ ]:
probability_only = StrategyRules(
    name='probability only',
    rules=(
        StrategyRule('sell', (MetricCondition('probability', '<=', 0.30),), 0.0),
        StrategyRule('buy', (MetricCondition('probability', '>=', 0.65),), 0.10),
    ),
)

probability_momentum = StrategyRules(
    name='probability plus momentum',
    rules=(
        StrategyRule('sell', (MetricCondition('probability', '<=', 0.30),), 0.0),
        StrategyRule(
            'buy',
            (MetricCondition('probability', '>=', 0.65), MetricCondition('momentum_score', '>=', 0.0)),
            0.10,
        ),
    ),
)
candidates = (probability_only, probability_momentum)

## Run the experiment

The first 50% of observations are training data, the next 25% are validation data, and the final 25% are the untouched test interval.

In [ ]:
split = ExperimentSplit.from_fractions(prices.index, train_fraction=0.50, validation_fraction=0.25)
experiment = run_strategy_experiment(
    prices,
    metrics,
    candidates,
    split,
    config=StrategyExperimentConfig(selection_metric='sharpe_ratio', initial_cash=10_000.0),
)
experiment.selected_strategy.name

In [ ]:
experiment.train_table

In [ ]:
experiment.validation_table

In [ ]:
experiment.test_table

The test table contains only the validation-selected strategy plus equal-weight and cash benchmarks. Other candidates are intentionally not evaluated on the test interval, preventing the test data from becoming another selection set.

In [ ]:
experiment.warnings